# Streaming & SSE

## Overview

This document explains how `stream_ollama_response()` works — an async generator that streams tokens from Ollama one at a time. This is what makes the chat UI feel responsive: instead of waiting 10+ seconds for a complete answer, users see words appearing as the LLM generates them.

## Architecture Context

In the chat service, `chain.py` uses `stream_ollama_response()` inside `rag_query()`. The `/chat` endpoint wraps this in an SSE (Server-Sent Events) response using `sse-starlette`. The frontend reads the SSE stream and appends each token to the message bubble.

In [ ]:
# Prerequisites — requires Ollama with mistral
import httpx
import json
import time

OLLAMA_BASE_URL = "http://localhost:11434"

async def check_ollama():
    try:
        async with httpx.AsyncClient() as client:
            resp = await client.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5.0)
            models = [m["name"] for m in resp.json().get("models", [])]
            has_mistral = any("mistral" in m for m in models)
            print(f"✓ Ollama: mistral={'✓' if has_mistral else '✗'}")
    except Exception as e:
        print(f"✗ Ollama: {e}")

await check_ollama()

## Package Introductions

No new packages — we're using httpx's streaming capabilities, which were introduced in `03_embeddings.ipynb`.

The new concept is **async generators** — Python's way of producing a stream of values lazily. If you've used Go channels to stream data between goroutines, async generators serve a similar purpose but with different syntax.

### Async Generators
```python
# Go channel pattern:
# ch := make(chan string)
# go func() { ch <- "hello"; ch <- "world"; close(ch) }()
# for msg := range ch { fmt.Println(msg) }

# Python async generator equivalent:
async def my_stream():
    yield "hello"
    yield "world"

async for msg in my_stream():
    print(msg)
```

Key difference: Go channels are concurrent (producer and consumer run in separate goroutines). Python async generators are cooperative — `yield` suspends the generator until the consumer asks for the next value with `async for`.

## Go/TS Comparison

| Concept | Go | Python |
|---------|-----|--------|
| Streaming values | `chan T` | `async def f(): yield value` |
| Consuming stream | `for v := range ch` | `async for v in f()` |
| Stream done | `close(ch)` | function returns (implicit) |
| HTTP streaming | `resp.Body` (io.Reader) | `response.aiter_lines()` |

In Go, you'd read a streaming HTTP response with `bufio.NewScanner(resp.Body)` and iterate lines. In Python, httpx gives you `response.aiter_lines()` which is an async iterator over lines as they arrive.

> **Pitfall:** Claude Code sometimes generates non-streaming Ollama calls (`"stream": false`) when streaming is needed. The non-streaming version waits for the entire response before returning, which defeats the purpose. Always set `"stream": true` when building the chat endpoint.

## Build It

### Step 1: Non-streaming baseline

First, let's see what a non-streaming call looks like and how long it takes.

In [ ]:
import httpx
import time

OLLAMA_BASE_URL = "http://localhost:11434"

# Non-streaming — wait for full response
start = time.time()
async with httpx.AsyncClient() as client:
    response = await client.post(
        f"{OLLAMA_BASE_URL}/api/generate",
        json={
            "model": "mistral",
            "prompt": "Explain what RAG means in AI, in 2 sentences.",
            "stream": False,
        },
        timeout=120.0,
    )
    data = response.json()
    elapsed = time.time() - start

print(f"Response ({elapsed:.1f}s wait):")
print(data["response"])
print(f"\nThe user sees NOTHING for {elapsed:.1f} seconds, then the full text appears at once.")

### Step 2: Streaming — see tokens arrive

Now let's stream the same request. The LLM sends back JSON objects line by line, each containing one token.

In [ ]:
import json

# Streaming — tokens arrive one at a time
print("Streaming response:")
start = time.time()
first_token_time = None

async with httpx.AsyncClient() as client:
    async with client.stream(
        "POST",
        f"{OLLAMA_BASE_URL}/api/generate",
        json={
            "model": "mistral",
            "prompt": "Explain what RAG means in AI, in 2 sentences.",
            "stream": True,
        },
        timeout=120.0,
    ) as response:
        async for line in response.aiter_lines():
            if line.strip():
                data = json.loads(line)
                if data.get("response"):
                    if first_token_time is None:
                        first_token_time = time.time() - start
                    print(data["response"], end="", flush=True)
                if data.get("done"):
                    break

elapsed = time.time() - start
print(f"\n\nFirst token: {first_token_time:.1f}s, Total: {elapsed:.1f}s")
print(f"The user starts reading after {first_token_time:.1f}s instead of waiting {elapsed:.1f}s.")

### Step 3: Build the reusable async generator

Now let's wrap this in a clean function — the `stream_ollama_response()` that the chat service uses.

In [ ]:
from typing import AsyncGenerator

SYSTEM_PROMPT = """You are a helpful document Q&A assistant. Answer questions based only on the provided context."""

async def stream_ollama_response(
    prompt: str,
    model: str,
    base_url: str,
) -> AsyncGenerator[dict, None]:
    """Stream tokens from Ollama as an async generator.

    Yields dicts like {"token": "The"} for each token.
    """
    async with httpx.AsyncClient() as client:
        async with client.stream(
            "POST",
            f"{base_url}/api/generate",
            json={
                "model": model,
                "prompt": prompt,
                "system": SYSTEM_PROMPT,
                "stream": True,
            },
            timeout=300.0,
        ) as response:
            response.raise_for_status()

            async for line in response.aiter_lines():
                if line.strip():
                    data = json.loads(line)
                    if data.get("response"):
                        yield {"token": data["response"]}
                    if data.get("done"):
                        break

# Use the generator
print("Using stream_ollama_response():")
async for event in stream_ollama_response(
    prompt="What is machine learning? One sentence.",
    model="mistral",
    base_url=OLLAMA_BASE_URL,
):
    print(event["token"], end="", flush=True)
print()

### Step 4: SSE format

When we serve this from FastAPI, each event gets wrapped in SSE format: `data: {"token": "..."}\n\n`. This is the format the browser's `EventSource` API (and our frontend's `fetch` + `ReadableStream`) expects.

Here's what the raw SSE stream looks like:

In [ ]:
# Simulate what the /chat endpoint sends to the browser
print("Raw SSE format:\n")
token_count = 0
async for event in stream_ollama_response(
    prompt="What is Python? One sentence.",
    model="mistral",
    base_url=OLLAMA_BASE_URL,
):
    sse_line = f"data: {json.dumps(event)}"
    print(sse_line)
    token_count += 1
    if token_count > 8:
        print("...")
        break

# Final event with sources
final = {"done": True, "sources": [{"file": "docs.pdf", "page": 1}]}
print(f"data: {json.dumps(final)}")
print("\nEach line is a separate SSE event. The frontend parses each 'data:' line as JSON.")

## Experiment

In [ ]:
# Experiment 1: Count tokens and measure throughput
token_count = 0
start = time.time()

async for event in stream_ollama_response(
    prompt="Write a short paragraph about vector databases.",
    model="mistral",
    base_url=OLLAMA_BASE_URL,
):
    token_count += 1

elapsed = time.time() - start
print(f"Tokens: {token_count}")
print(f"Time: {elapsed:.1f}s")
print(f"Speed: {token_count/elapsed:.1f} tokens/sec")

## Check Your Understanding

1. **Why stream responses instead of waiting for the complete text? What's the UX difference?**

2. **How is a Python async generator different from a Go channel? What does `yield` do?**

3. **Why does the SSE stream end with a `{"done": true, "sources": [...]}` event instead of just stopping?**